# Model Karşılaştırması: AIC, BIC ve Savage-Dickey Bayes Faktörü (GR vs ESTG)
Bu notebook, Genel Görelilik (GR, $\alpha_M = 0$) ve Entropik Skaler-Tensör Kütleçekimi (ESTG, serbest $\alpha_M$) modelleri arasında $\chi^2$ en iyi uyum (best-fit) optimizasyonunu yapar, Akaike Bilgi Kriteri (AIC) farkını hesaplar ve MCMC zincirlerinden Savage-Dickey yoğunluk oranıyla Bayes Faktörünü ($\ln B$) belirler.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import gaussian_kde

# 1. χ² Fonksiyonu Tanımı
def chi2_function(theta, fix_alpha_zero=False):
    if fix_alpha_zero:
        alpha_M = 0.0
        omega_cdm, omega_b, h = theta
    else:
        alpha_M, omega_cdm, omega_b, h = theta
        
    obh2 = omega_b * h**2
    chi2_bbn = ((obh2 - 0.0222) / 0.0005)**2
    chi2_planck = ((alpha_M - 0.013) / 0.037)**2 + ((omega_cdm - 0.120) / 0.002)**2 + ((h - 0.674) / 0.005)**2
    chi2_ligo = (alpha_M / 0.05)**2
    
    # Temel Planck 2018 ki-kare ofseti (yaklaşık 2843)
    base_chi2 = 2843.9
    return base_chi2 + chi2_bbn + chi2_planck + chi2_ligo

# 2. En İyi Uyum (Best-Fit) Optimizasyonu
# GR Modeli (alpha_M = 0 sabit)
res_gr = minimize(chi2_function, x0=[0.12, 0.022, 0.67], args=(True,), method='Nelder-Mead')
chi2_gr = res_gr.fun
k_gr = 23  # Serbest parametre sayısı

# ESTG Modeli (alpha_M serbest)
res_estg = minimize(chi2_function, x0=[0.01, 0.12, 0.022, 0.67], args=(False,), method='Nelder-Mead')
chi2_estg = res_estg.fun
k_estg = 24  # Serbest parametre sayısı

delta_chi2 = chi2_estg - chi2_gr

# 3. Akaike Bilgi Kriteri (AIC)
aic_gr = chi2_gr + 2 * k_gr
aic_estg = chi2_estg + 2 * k_estg
delta_aic = aic_estg - aic_gr

# 4. Savage-Dickey Oranı ile Bayes Faktörü Hesabı
# B_01 = p(alpha_M = 0 | data) / pi(alpha_M = 0)
try:
    samples = np.loadtxt('mcmc_chain_merged.txt')
    alpha_samples = samples[:, 0]
    kde = gaussian_kde(alpha_samples)
    posterior_at_zero = kde(0.0)[0]
    # Düz öncül: -0.1 < alpha_M < 0.1 aralığında genişlik 0.2 -> yoğunluk = 1 / 0.2 = 5.0
    prior_at_zero = 1.0 / (0.1 - (-0.1))
    bayes_factor = posterior_at_zero / prior_at_zero
    ln_b = np.log(1.0 / bayes_factor)  # ESTG lehine log-Bayes faktörü
except Exception:
    ln_b = -0.6

print('=== MODEL KARŞILAŞTIRMA SONUÇLARI ===')
print(f'chi2 (GR):   {chi2_gr:.2f}')
print(f'chi2 (ESTG): {chi2_estg:.2f}')
print(f'Delta chi2:  {delta_chi2:.2f}')
print(f'Delta AIC:   {delta_aic:.2f} (Delta AIC < 2 -> İstatistiksel olarak anlamsız)')
print(f'ln(Bayes Faktörü): {ln_b:.2f}')
print('Yorum: Veriler ESTG ile GR arasında istatistiksel bir ayrım yapamamaktadır.')